# Mô hình CRF cho bài toán NER (Named Entity Recognition)
## VLSP 2016 Dataset - Tiếng Việt

---

## 1. Cài đặt thư viện

In [1]:
import ast # đọc file dataset dạng dict
from pathlib import Path # xử lí đường dẫn
import joblib # lưu mô hình đã train vào file .jolib
import pandas as pd # dạng bảng dữ liệu
import sklearn_crfsuite # thư viện CRF để train mô hình
from sklearn.metrics import classification_report, precision_recall_fscore_support # đánh giá mô hình
from underthesea import word_tokenize # thư viện tách từ tiếng Việt, 


## 2. Định nghĩa Label Mapping

In [2]:

LABEL_MAP = {
    0: "O",
    # VLSP 2016 sử dụng 8 nhãn thực thể
    1: "B-PER",
    2: "I-PER",
    3: "B-ORG",
    4: "I-ORG",
    5: "B-LOC",
    6: "I-LOC",
    7: "B-MISC",
    8: "I-MISC",
}
# đọc dữ liệu 5 dòng dầu tiên để kiểm tra
data_path = Path("data/vlsp_train_raw.csv")
df = pd.read_csv(data_path)
print(df.head())

                                              tokens  \
0            ["Không_khí", "thật", "náo_nhiệt", "."]   
1  ["Chị", "Lãnh", "và", "Xăng", "ra", "đi", ",",...   
2  ["Suy_tính", "mãi", ",", "khóc", "mãi", "rồi",...   
3  ["Hoà", "bảo", "hồi", "mới", "qua", "đâu", "có...   
4             ["Nhật_ký", "của", "thuyền_viên", "."]   

                                            ner_tags  
0                                       [0, 0, 0, 0]  
1  [0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...  
2  [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, ...  
3  [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, ...  
4                                       [0, 0, 0, 0]  


DATA đã được word segmentation , gán nhãn 

## 3. Định nghĩa Feature Engineering

In [3]:
def word2features(sent, i):
    """
    Trích xuất features cho một từ trong câu
    """
    # từ hiện tại ("Ông", "O") -> 0 là ông, 1 là O
    word = sent[i][0]
    
    features = {
        "bias": 1.0, # đặc trưng cố định để giúp mô hình học tốt hơn
        "word.lower()": word.lower(), 
        "word.istitle()": word.istitle(), # viết đầu chữ cái hoa
        "word.isupper()": word.isupper(), 
        "word.isdigit()": word.isdigit(),
        "word_len": len(word), 
    }

    # Các prefix đặc trưng cho từng loại entity
    person_prefixes = {"ông", "bà", "anh", "chị", "em", "ngài", "thông", "nạn_nhân", "thợ", "cụ"}
    loc_prefixes = {"tại", "ở", "xã", "huyện", "tỉnh", "thành_phố", "quận", "khu_vực", "biển", "đảo", "phường"}
    org_prefixes = {"công_ty", "tập_đoàn", "bộ", "ban", "ngành", "ngân_hàng", "đại_học", "trường"}

    # tức là ko phải từ đầu câu , từ đứng trước đó
    if i > 0:
        word_prev = sent[i - 1][0].lower()
        features.update(
            {
                "-1:word.lower()": word_prev,
                "-1:word.istitle()": sent[i - 1][0].istitle(),
                "-1:is_person_prefix": word_prev in person_prefixes,
                "-1:is_loc_prefix": word_prev in loc_prefixes,
                "-1:is_org_prefix": word_prev in org_prefixes,
            }
        )
    else:
        features["BOS"] = True  # từ đầu tiên ko có từ đứng trc

    # Features của từ tiếp theo
    if i < len(sent) - 1:
        word_next = sent[i + 1][0].lower()
        features.update({
            "+1:word.lower()": word_next, 
            "+1:word.istitle()": sent[i + 1][0].istitle()}
            )
    else:
        features["EOS"] = True  # End of sentence
    return features

# trích xuất features cho mỗi từ trong câu
def sent2features(sent):
    features = []
    for i in range(len(sent)):
        f = word2features(sent, i)
        features.append(f)
    return features

## 4. Load dữ liệu từ CSV

In [4]:
def prepare_data_from_csv(csv_path):
    """
    Đọc dữ liệu từ CSV và chuyển đổi sang định dạng phù hợp cho CRF
    Trả về: sentence = [(token, BIO_TAG), ...]
    """
    df = pd.read_csv(csv_path)
    sentences = []

    for _, row in df.iterrows():
        # Parse tokens và tags từ string
        tokens = ast.literal_eval(row["tokens"]) if isinstance(row["tokens"], str) else row["tokens"]
        tags = ast.literal_eval(row["ner_tags"]) if isinstance(row["ner_tags"], str) else row["ner_tags"]

        # Chuyển đổi từ ID sang BIO tag
        words_and_tags = [(w, LABEL_MAP[tid]) for w, tid in zip(tokens, tags)]
        sentences.append(words_and_tags)

    return sentences

# Load dữ liệu train và validation
train_csv = Path("data/vlsp_train_raw.csv")
valid_csv = Path("data/vlsp_valid_raw.csv")

train_sentences = prepare_data_from_csv(train_csv)
valid_sentences = prepare_data_from_csv(valid_csv)

print(f"Số câu train: {len(train_sentences)}")
print(f"Số câu validation: {len(valid_sentences)}")
print(f"\nVí dụ câu đầu tiên trong tập train:")
print(train_sentences[0][:5])

Số câu train: 13486
Số câu validation: 3372

Ví dụ câu đầu tiên trong tập train:
[('Không_khí', 'O'), ('thật', 'O'), ('náo_nhiệt', 'O'), ('.', 'O')]


## 5. Trích xuất Features

In [5]:
print("Đang trích xuất features cho tập train...")
X_train = [sent2features(s) for s in train_sentences]
y_train = [[label for _, label in s] for s in train_sentences]

print("Đang trích xuất features cho tập validation...")
X_valid = [sent2features(s) for s in valid_sentences]
y_valid = [[label for _, label in s] for s in valid_sentences]

print(f"\nTrain: {len(X_train)} câu")
print(f"Valid: {len(X_valid)} câu")

"""
INPUT MODEL
[
  {
    "word.lower()": "ông",
    "BOS": True,
    ...
  },
  {
    "word.lower()": "nguyễn",
    "-1:word.lower()": "ông",
    "-1:is_person_prefix": True,
    ...
  }
]
"""

Đang trích xuất features cho tập train...
Đang trích xuất features cho tập validation...

Train: 13486 câu
Valid: 3372 câu


'\nINPUT MODEL\n[\n  {\n    "word.lower()": "ông",\n    "BOS": True,\n    ...\n  },\n  {\n    "word.lower()": "nguyễn",\n    "-1:word.lower()": "ông",\n    "-1:is_person_prefix": True,\n    ...\n  }\n]\n'

## 6. Huấn luyện mô hình CRF

In [6]:
# Khởi tạo và huấn luyện CRF
crf = sklearn_crfsuite.CRF(
    algorithm='lbfgs',
    c1=0.1,  # L1 regularization ép weights về 0
    c2=0.1,  # L2 regularization ep weights về nhỏ 
    max_iterations=100, # số lần lặp tối đa để huấn luyện = epoch
    all_possible_transitions=True, # cho học transition ko có trong train
)
# score = w1*f1 + w2*f2 + w3*f3 + ...
# nếu w2 = 0 thì f2 ko còn tác dụng nữa tránh học nhiều features không cần thiết
print("Bắt đầu huấn luyện CRF...")
crf.fit(X_train, y_train)
print("Huấn luyện hoàn tất!")

Bắt đầu huấn luyện CRF...
Huấn luyện hoàn tất!


## 7. Đánh giá mô hình

In [7]:
# Dự đoán trên tập validation
y_pred = crf.predict(X_valid)

# Chuyển đổi danh sách labels phẳng (flat)
y_true_flat = [label for sent in y_valid for label in sent]
y_pred_flat = [label for sent in y_pred for label in sent]

print("=" * 60)
print("KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH CRF")
print("=" * 60)

# Tính Precision, Recall, F1-Score cho từng nhãn
labels = list(LABEL_MAP.values())
labels.remove('O')  # Loại bỏ nhãn O để xem chi tiết các entity

print("\n📊 Báo cáo chi tiết từng loại entity:")
print("-" * 60)
print(classification_report(y_true_flat, y_pred_flat, labels=labels, zero_division=0))

KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH CRF

📊 Báo cáo chi tiết từng loại entity:
------------------------------------------------------------
              precision    recall  f1-score   support

       B-PER       0.96      0.96      0.96      1502
       I-PER       0.93      0.96      0.95       692
       B-ORG       0.84      0.59      0.69       266
       I-ORG       0.82      0.67      0.74       477
       B-LOC       0.92      0.89      0.90      1314
       I-LOC       0.88      0.82      0.85       594
      B-MISC       0.86      0.94      0.90        54
      I-MISC       0.86      0.91      0.89        56

   micro avg       0.92      0.88      0.90      4955
   macro avg       0.89      0.84      0.86      4955
weighted avg       0.91      0.88      0.89      4955



nhận xét : person tốt , org kém , location tốt ,

precision cao (~0.84)
recall thấp (~0.59)
Là : 

model đoán ORG khá đúng
NHƯNG bỏ sót rất nhiều ORG

 misc cx được nhưng ít data quá có 50 data
 tổ chức nó đa dạng hơn nhiều nên kết quả chưa tốt lắm
 

In [8]:
# Tính các metrics tổng quát
precision, recall, f1, support = precision_recall_fscore_support(
    y_true_flat, y_pred_flat, average='weighted', zero_division=0
)

print("\n📈 Các chỉ số tổng quát (Weighted Average):")
print("-" * 60)
print(f"  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  F1-Score:  {f1:.4f}")

# Tính cho từng loại entity
entity_types = ['PER', 'ORG', 'LOC', 'MISC']
print("\n📊 Chi tiết từng loại Entity:")
print("-" * 60)

for ent_type in entity_types:
    b_label = f"B-{ent_type}"
    i_label = f"I-{ent_type}"
    
    # Lọc các vị trí có entity thực sự
    true_entities = [1 if y in [b_label, i_label] else 0 for y in y_true_flat]
    pred_entities = [1 if y in [b_label, i_label] else 0 for y in y_pred_flat]
    
    if sum(true_entities) > 0:
        p, r, f, _ = precision_recall_fscore_support(true_entities, pred_entities, average='binary', zero_division=0)
        print(f"  {ent_type:6s} - Precision: {p:.4f}, Recall: {r:.4f}, F1: {f:.4f}")


📈 Các chỉ số tổng quát (Weighted Average):
------------------------------------------------------------
  Precision: 0.9897
  Recall:    0.9901
  F1-Score:  0.9897

📊 Chi tiết từng loại Entity:
------------------------------------------------------------
  PER    - Precision: 0.9530, Recall: 0.9622, F1: 0.9576
  ORG    - Precision: 0.8365, Recall: 0.6474, F1: 0.7299
  LOC    - Precision: 0.9311, Recall: 0.8920, F1: 0.9111
  MISC   - Precision: 0.8729, Recall: 0.9364, F1: 0.9035


ORG nó đang thấp nhất 
 Nhận diện người (PER): rất tốt
 Địa điểm (LOC): tốt
 MISC: ổn
 Tổ chức (ORG): yếu (thiếu recall)


 

<h1>8 Sửa lại hàm word2features cho ORG cao lên </h1>

In [9]:
def word2features(sent, i):
    word = sent[i][0]

    # ====== FEATURE CƠ BẢN ======
    features = {
        "bias": 1.0,
        "word.lower()": word.lower(),
        "word.istitle()": word.istitle(),
        "word.isupper()": word.isupper(),
        "word.isdigit()": word.isdigit(),
        "word_len": len(word),

        # lấy kí tự đầu cuối
        "prefix_2": word[:2],
        "prefix_3": word[:3],
        "suffix_2": word[-2:],
        "suffix_3": word[-3:],

        #  chứa số không vì số nhà các kiểu
        "has_digit": any(c.isdigit() for c in word),

        # shape của từ: viết hoa, viết thường, chữ số, ký tự đặc biệt 
        # "Hà_Nội" -> "Xx_Xxxx" tức là tạo partern chung cho các từ có cùng kiểu viết
        "word_shape": "".join(
            "X" if c.isupper() else
            "x" if c.islower() else
            "d" if c.isdigit() else c
            for c in word
        ),
    }

    # ====== PREFIX ENTITY ======
    person_prefixes = {"ông", "bà", "anh", "chị", "em", "ngài", "thông", "nạn_nhân", "thợ", "cụ"} 
    loc_prefixes = {"tại", "ở", "xã", "huyện", "tỉnh", "thành_phố", "quận", "khu_vực", "biển", "đảo", "phường"} 
    org_prefixes = {"công_ty", "tập_đoàn", "bộ", "ban", "ngành", "ngân_hàng", "đại_học", "trường"}
    # ====== CONTEXT TRÁI ======
    if i > 0:
        word_prev = sent[i - 1][0].lower()

        features.update({
            "-1:word.lower()": word_prev,
            "-1:word.istitle()": sent[i - 1][0].istitle(),

            "-1:is_person_prefix": word_prev in person_prefixes,
            "-1:is_loc_prefix": word_prev in loc_prefixes,
            "-1:is_org_prefix": word_prev in org_prefixes,

            #  : bigram trái
            "-1:bigram": word_prev + "_" + word.lower() #"ông_nguyễn" → PERSON 
        })
    else:
        features["BOS"] = True

    # ====== CONTEXT PHẢI ======
    if i < len(sent) - 1:
        word_next = sent[i + 1][0].lower()

        features.update({
            "+1:word.lower()": word_next,
            "+1:word.istitle()": sent[i + 1][0].istitle(),

            #  bigram phải
            "+1:bigram": word.lower() + "_" + word_next
        })
    else:
        features["EOS"] = True

    return features


def sent2features(sent):
    return [word2features(sent, i) for i in range(len(sent))]
def prepare_data_from_csv(csv_path):
    """
    Đọc dữ liệu từ CSV và chuyển đổi sang định dạng phù hợp cho CRF
    Trả về: sentence = [(token, BIO_TAG), ...]
    """
    df = pd.read_csv(csv_path)
    sentences = []

    for _, row in df.iterrows():
        # Parse tokens và tags từ string
        tokens = ast.literal_eval(row["tokens"]) if isinstance(row["tokens"], str) else row["tokens"]
        tags = ast.literal_eval(row["ner_tags"]) if isinstance(row["ner_tags"], str) else row["ner_tags"]

        # Chuyển đổi từ ID sang BIO tag
        words_and_tags = [(w, LABEL_MAP[tid]) for w, tid in zip(tokens, tags)]
        sentences.append(words_and_tags)

    return sentences

# Load dữ liệu train và validation
train_csv = Path("data/vlsp_train_raw.csv")
valid_csv = Path("data/vlsp_valid_raw.csv")

train_sentences = prepare_data_from_csv(train_csv)
valid_sentences = prepare_data_from_csv(valid_csv)

print(f"Số câu train: {len(train_sentences)}")
print(f"Số câu validation: {len(valid_sentences)}")
print(f"\nVí dụ câu đầu tiên trong tập train:")
print(train_sentences[0][:5])
print("Đang trích xuất features cho tập train...")
X_train = [sent2features(s) for s in train_sentences]
y_train = [[label for _, label in s] for s in train_sentences]

print("Đang trích xuất features cho tập validation...")
X_valid = [sent2features(s) for s in valid_sentences]
y_valid = [[label for _, label in s] for s in valid_sentences]

print(f"\nTrain: {len(X_train)} câu")
print(f"Valid: {len(X_valid)} câu")

Số câu train: 13486
Số câu validation: 3372

Ví dụ câu đầu tiên trong tập train:
[('Không_khí', 'O'), ('thật', 'O'), ('náo_nhiệt', 'O'), ('.', 'O')]
Đang trích xuất features cho tập train...
Đang trích xuất features cho tập validation...

Train: 13486 câu
Valid: 3372 câu


<h2>8.1 Chạy mô hình</h2>

In [10]:
# Khởi tạo và huấn luyện CRF
crf = sklearn_crfsuite.CRF(
    algorithm='lbfgs',
    c1=0.1,  # L1 regularization ép weights về 0
    c2=0.1,  # L2 regularization ep weights về nhỏ 
    max_iterations=100,
    all_possible_transitions=True,
)
# score = w1*f1 + w2*f2 + w3*f3 + ...
# nếu w2 = 0 thì f2 ko còn tác dụng nữa tránh học nhiều features không cần thiết
print("Bắt đầu huấn luyện CRF...")
crf.fit(X_train, y_train)
print("Huấn luyện hoàn tất!")

Bắt đầu huấn luyện CRF...
Huấn luyện hoàn tất!


In [11]:
# Dự đoán trên tập validation
y_pred = crf.predict(X_valid)

# Chuyển đổi danh sách labels phẳng (flat)
y_true_flat = [label for sent in y_valid for label in sent]
y_pred_flat = [label for sent in y_pred for label in sent]

print("=" * 60)
print("KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH CRF")
print("=" * 60)

# Tính Precision, Recall, F1-Score cho từng nhãn
labels = list(LABEL_MAP.values())
labels.remove('O')  # Loại bỏ nhãn O để xem chi tiết các entity

print("\n📊 Báo cáo chi tiết từng loại entity:")
print("-" * 60)
print(classification_report(y_true_flat, y_pred_flat, labels=labels, zero_division=0))

KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH CRF

📊 Báo cáo chi tiết từng loại entity:
------------------------------------------------------------
              precision    recall  f1-score   support

       B-PER       0.96      0.96      0.96      1502
       I-PER       0.94      0.97      0.96       692
       B-ORG       0.87      0.68      0.76       266
       I-ORG       0.85      0.77      0.81       477
       B-LOC       0.93      0.90      0.91      1314
       I-LOC       0.91      0.84      0.87       594
      B-MISC       0.91      0.96      0.94        54
      I-MISC       0.91      0.93      0.92        56

   micro avg       0.93      0.90      0.91      4955
   macro avg       0.91      0.88      0.89      4955
weighted avg       0.93      0.90      0.91      4955



TỐT hơn là rõ khi tăng lên 78 81 rồi

## 9. Lưu mô hình

In [12]:
# Lưu model
def save_crf_artifact(model, out_path: Path) -> None:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump({"model": model}, out_path)

model_out = Path("models/crf_vlsp2016_notebook.joblib")
save_crf_artifact(crf, model_out)
print(f"Đã lưu CRF model vào: {model_out}")

Đã lưu CRF model vào: models\crf_vlsp2016_notebook.joblib


## 10. Ví dụ dự đoán

In [13]:
# Ví dụ dự đoán một câu
print("\n" + "=" * 60)
print("VÍ DỤ DỰ ĐOÁN")
print("=" * 60)

sample_idx = 0
sample_sent = valid_sentences[sample_idx]
sample_pred = y_pred[sample_idx]

print("\nCâu:\n")
for i, (word, true_tag) in enumerate(sample_sent):
    pred_tag = sample_pred[i]
    marker = "✓" if true_tag == pred_tag else "✗"
    print(f"  {word:15s} | True: {true_tag:8s} | Pred: {pred_tag:8s} {marker}")


VÍ DỤ DỰ ĐOÁN

Câu:

  Nào             | True: O        | Pred: O        ✓
  ngồi            | True: O        | Pred: O        ✓
  xuống           | True: O        | Pred: O        ✓
  .               | True: O        | Pred: O        ✓


---
## Tổng kết Mô hình CRF

 **Ưu điểm:**
- Code đơn giản, dễ đọc và hiểu
- Huấn luyện nhanh
- Không cần GPU
- Feature engineering linh hoạt

 **Nhược điểm:**
- Phụ thuộc nhiều vào feature engineering
- Không học được các pattern phức tạp
- Recall có thể thấp hơn các mô hình deep learning